# Exercise: judge flag classification

## Background

`SLO-Nemotron-personas` generates synthetic Slovene personas in two stages:

1. **Generation.** 1,000 fixed demographic records (name, age, region, occupation, marital status, a Big Five personality profile, etc.) are each handed to three different LLMs — DeepSeek, GaMS-2, Gemma-3 — who independently write four blocks of free text per person: cultural background, skills & expertise, career goals, hobbies & interests.
2. **Judging.** An LLM judge reads each generation against its record and flags specific spans of text, each tagged with one of four marks:
   - **contradiction** — states something the record says is false (e.g. claims a job when the record says unemployed)
   - **invention** — adds biographical detail that isn't in the record, but isn't contradicted by it either (allowed, even expected)
   - **language** — a grammar, agreement, or wording error, unrelated to the record
   - **correct** — a case handled well that could easily have gone wrong (e.g. correctly acknowledging retirement instead of inventing a job)


## Your task

Build the training set for this from the raw source files, then finetune SloBERTa to classify a flagged span into one of the four marks above, giving it the person's record alongside the span as context. 


## 1. Setup

Install `transformers`, `accelerate`, `datasets`, `evaluate`, `scikit-learn` and `pandas`.

In [ ]:
!pip install -q transformers accelerate datasets evaluate scikit-learn pandas

## 2. Get the raw data

The raw files live in the same repo as this notebook. Get them into your working directory:

```text
personas_sl_deepseek_multiple.jsonl
personas_sl_gams2_multiple.jsonl
personas_sl_gemma3_multiple.jsonl
judge_output/
├── kimi_deepseek_analysis.jsonl
├── kimi_gams2_analysis.jsonl
└── kimi_gemma3_analysis.jsonl
```

## 3. Build the dataset

### The files

All files are JSON Lines (one JSON object per line).

**Personas**, `personas_sl_{generator}_multiple.jsonl`, hold one record per generation, identified by `(uuid, generation_index)`. The fields you'll need:

| field | meaning |
|---|---|
| `first_name`, `last_name`, `sex`, `age` | basic demographics |
| `marital_status`, `household_status` | family situation |
| `education_level`, `bachelors_field` | education (`bachelors_field` contains `"brez diplome"` if there is no degree) |
| `activity_status` | work status (employed, retired, unemployed, …) |
| `occupation`, `detailed_occupation` | job; may be empty |
| `statistical_region`, `municipality`, `post_code`, `post_town`, `settlement` | location |
| `openness`, `conscientiousness`, `extraversion`, `agreeableness`, `neuroticism` | Big Five traits, each a dict with a `label` key |

**Judge verdicts**, `judge_output/kimi_{generator}_analysis.jsonl`, hold one verdict per generation, with `uuid`, `generation_index` and a list of `spans`. Each span has `text`, `block`, `mark` (this is the label), `covers` and `field`.

### What to build

A table with **one row per flagged span**, containing the span text, the person's record as context, and the mark. Rules:

- skip spans that stand for a whole bullet list (`covers == "bullet_list"`), since those are list fragments rather than sentences
- attach the same context to every span of a generation, regardless of its mark: the person's record fields plus their Big Five profile, turned into a single string
- **don't** use the judge's own per-span `field` annotation as context — the rubric only ever fills that in for `contradiction` spans, so using it as a model input would leak the label
- skip verdicts with no matching persona record, and drop duplicate spans

<details><summary>Hint: matching verdicts to personas</summary>

For each generator, build a dict from `(uuid, generation_index)` to the persona record, then look up each verdict in it.
</details>

<details><summary>Hint: the context string</summary>

Keep it simple and readable, for example:
`Ime: Nada Hren | Spol: ženski | Starost: 67 | ... | Poklic: ... | OCEAN: openness=visoko, ...`

Leave out empty fields like a missing occupation, rather than writing `None`.
</details>

<details><summary>Hint: columns to keep</summary>

Besides the text, context and label, keep `uuid`, because you'll need it for the split.
</details>

Check what you built: how many rows do you have, and how are the labels distributed? Read a few rows and make sure the context makes sense.

## 4. Train/validation split

Make an 80/20 split **by person (`uuid`), not by row**. Each person's record appears in many rows, across 3 generators and several spans each. A random split would put the same people in both train and validation. 

Before splitting:
- drop classes with fewer than 5 examples
- map the labels to integer ids

<details><summary>Hint</summary>

`sklearn.model_selection.GroupShuffleSplit(n_splits=1, test_size=0.2)` with `groups=df["uuid"]`. Afterwards, check that no uuid appears in both splits.
</details>

## 5. Train

Finetune `EMBEDDIA/sloberta` for sequence classification. Each input is the context string, then `[SEP]`, then the span text.

Suggested settings: `max_length=256`, 3 epochs, learning rate `2e-5`, batch size 16, weight decay 0.01, 50 warmup steps, a cosine schedule with `min_lr=2e-6`, evaluation once per epoch, bf16. Track accuracy and macro-F1.

<details><summary>Hint: the pieces you need</summary>

`AutoTokenizer`, `AutoModelForSequenceClassification`, `datasets.Dataset.from_dict`, `DataCollatorWithPadding`, `TrainingArguments`, `Trainer`. Name the label column `labels`, since that's what the Trainer expects.
</details>

<details><summary>Hint: reproducibility</summary>

Call `transformers.set_seed(...)` right before you load the model, so the random weights of the classification head are the same on every run.
</details>

## 6. Evaluate

Print a per-class classification report and a confusion matrix for the validation set. Which marks are easy, and which get confused with each other?

## Bonus

- Replace `GroupShuffleSplit` with `StratifiedGroupKFold`, which keeps the split grouped by person and also balances the classes. Does it change the scores for the rare classes?